In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

In [3]:
portfolio = pd.read_csv('../../../integrated/data/portfolio.csv')
profile = pd.read_csv('../../../integrated/data/profile.csv')
transcript = pd.read_csv('../../../integrated/data/transcript.csv')
menu = pd.read_csv('../../../integrated/data/starbucks_menu_260112.csv')

## 고객 세그먼트별 기준 설정

- 일단은 고객 세그먼트는 구매 기반을을 기준으로 신규, 기존, VIP 고객으로 구분하여 탐색을 진행해볼게요.
# VIP 고객은 구매 금액 상위 20%이면서 동시에 구매 횟수 또한 높은 고객으로 정의할 생각

# 핵심 컬럼 정리(transcript)
- person: 고객 ID	
- event: record에 대한 정보 (이벤트 유형)	- 거래(transaction)
- 프로모션 받음(offer received)
- 프로모션 확인(offer viewed)
- 프로모션 완료(offer completed)
- value: 이벤트별 상세 정보	- 거래(transaction): 거래 금액(amount)
- 프로모션 이벤트: 프로모션 ID(offer id)
- 일부 거래 이벤트에는 거래 id나 보너스 여부 같은 추가 key가 포함될 수도 있음
- time: 시간 정보 (데이터 내 상대적 시간, t=0부터 시작)	

In [4]:
# 구매금액부터 실행
transcript['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

In [5]:
# 거래컬럼 추출
transaction = transcript[transcript['event'] == 'transaction'].copy()

In [6]:
transaction.isna().sum()

Unnamed: 0    0
person        0
event         0
value         0
time          0
dtype: int64

In [7]:
transaction.describe()

,Unnamed: 0,time
count,138953.000000,138953.000000
mean,160710.678676,381.584334
std,90463.542633,201.697230
min,12654.000000,0.000000
25%,85177.000000,210.000000
50%,150201.000000,402.000000
75%,239167.000000,552.000000
max,306533.000000,714.000000


In [8]:
transaction

,Unnamed: 0,person,event,value,time
12654,12654,02c083884c7d45b39cc68e1314fec56c,transaction,{'amount': 0.8300000000000001},0
12657,12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,transaction,{'amount': 34.56},0
12659,12659,54890f68699049c2a04d415abc25e717,transaction,{'amount': 13.23},0
12670,12670,b2f1cd155b864803ad8334cdf13c4bd2,transaction,{'amount': 19.51},0
12671,12671,fe97aa22dd3e48c8b143116a8403dd52,transaction,{'amount': 18.97},0
...,...,...,...,...,...
306529,306529,b3a1272bc9904337b331bf348c3e8c17,transaction,{'amount': 1.5899999999999999},714
306530,306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,{'amount': 9.53},714
306531,306531,a00058cf10334a308c68e7631c529907,transaction,{'amount': 3.61},714
306532,306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,{'amount': 3.5300000000000002},714


In [9]:
# 금액이 value컬럼에 있어 확인(문자열 형식이 str이라 dict로)
transaction['value'].head()

12654    {'amount': 0.8300000000000001}
12657                 {'amount': 34.56}
12659                 {'amount': 13.23}
12670                 {'amount': 19.51}
12671                 {'amount': 18.97}
Name: value, dtype: str

In [10]:
# 문자열을 dict로 변환할때
import ast
transaction['value'] = transaction['value'].apply(ast.literal_eval)

In [11]:
type(transaction['value'].iloc[0])

dict

In [12]:
transaction['value'].head()

12654    {'amount': 0.8300000000000001}
12657                 {'amount': 34.56}
12659                 {'amount': 13.23}
12670                 {'amount': 19.51}
12671                 {'amount': 18.97}
Name: value, dtype: object

In [13]:
# 금액컬럼 추출
transaction['amount'] = transaction['value'].apply(lambda x: x['amount'])

In [14]:
transaction.head()

,Unnamed: 0,person,event,value,time,amount
12654,12654,02c083884c7d45b39cc68e1314fec56c,transaction,{'amount': 0.8300000000000001},0,0.83
12657,12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,transaction,{'amount': 34.56},0,34.56
12659,12659,54890f68699049c2a04d415abc25e717,transaction,{'amount': 13.23},0,13.23
12670,12670,b2f1cd155b864803ad8334cdf13c4bd2,transaction,{'amount': 19.51},0,19.51
12671,12671,fe97aa22dd3e48c8b143116a8403dd52,transaction,{'amount': 18.97},0,18.97


In [15]:
transaction['amount'].value_counts()

amount
0.05      431
0.66      166
1.18      165
1.01      163
1.23      161
         ... 
290.93      1
43.91       1
685.07      1
405.04      1
476.33      1
Name: count, Length: 5103, dtype: int64

In [16]:
transaction['amount'].describe()

count    138953.000000
mean         12.777356
std          30.250529
min           0.050000
25%           2.780000
50%           8.890000
75%          18.070000
max        1062.280000
Name: amount, dtype: float64

스타벅스에서 max값이 1062달러?? 가능한가??

In [17]:
transaction.sort_values('amount', ascending=False).head(60)

,Unnamed: 0,person,event,value,time,amount
284273,284273,6cf884a6c5ae4b2daccb6d3108028fef,transaction,{'amount': 1062.28},630,1062.28
301790,301790,454b00bdd77c4f588eb9f6cafd81dc5d,transaction,{'amount': 1015.73},690,1015.73
42253,42253,5ecaea5c8bf44ae4a65838d9d066c909,transaction,{'amount': 1001.85},102,1001.85
140346,140346,dce784e26f294101999d000fad9089bb,transaction,{'amount': 989.34},372,989.34
302951,302951,52959f19113e4241a8cb3bef486c6412,transaction,{'amount': 977.78},696,977.78
294112,294112,626df8678e2a4953b9098246418c9cfa,transaction,{'amount': 966.24},660,966.24
21490,21490,c7a5d7bab83a4d82a7f977b3d586f3d4,transaction,{'amount': 962.1},18,962.10
280015,280015,6406abad8e2c4b8584e4f68003de148d,transaction,{'amount': 961.21},618,961.21
280202,280202,ef65dd061ff24b0f805a51dd0328dbcb,transaction,{'amount': 957.15},618,957.15
27692,27692,a73cf044395d46ea804f688490ad9227,transaction,{'amount': 947.43},36,947.43


극단값이라고 생각든게 900~1000달러 기준으로 여러개 있음.... 여러구매를 합한거 같음 누적이겠지?

그래서 고가를 기준으로 vip 선정하돼 한번에 쏜사람이 포함될수도 있으니 구매횟수까지 포함해서 vip 선정 해야될듯

In [18]:
# 일단 금액확인했으니 동일 기준id로 구매횟수랑 금액 집계
customer = transaction.groupby('person').agg({
    'amount': ['count', 'sum']
}).reset_index()

In [19]:
customer.head()

person amount        
                                     count     sum
0  0009655768c64bdeb2e877511632db8f      8  127.60
1  00116118485d4dfda04fdbaba9a87b5c      3    4.09
2  0011e0d4e6b944f998e987f904e8c1e5      5   79.46
3  0020c2b971eb4e9188eac86d93036a77      8  196.86
4  0020ccbbb6d84e358d3414a3ff76cffd     12  154.05

In [20]:
# 구매횟수랑 총금액 컬럼 정의
customer.columns = ['person', 'purchase_count', 'total_amount']

In [21]:
# 동일id기준 구매횟수랑 총금액 확인
customer.head()

,person,purchase_count,total_amount
0,0009655768c64bdeb2e877511632db8f,8,127.60
1,00116118485d4dfda04fdbaba9a87b5c,3,4.09
2,0011e0d4e6b944f998e987f904e8c1e5,5,79.46
3,0020c2b971eb4e9188eac86d93036a77,8,196.86
4,0020ccbbb6d84e358d3414a3ff76cffd,12,154.05


In [22]:
customer.tail()

,person,purchase_count,total_amount
16573,fff3ba4757bd42088c044ca26d73817a,11,580.98
16574,fff7576017104bcc8677a8d63322b5e1,6,29.94
16575,fff8957ea8b240a6b5e634b6ee8eafcf,5,12.15
16576,fffad4f4828548d1b5583907f2e9906b,12,88.83
16577,ffff82501cea40309d5fdd7edcca4a07,15,226.07


In [23]:
customer.sort_values('total_amount', ascending=False).head(10)

,person,purchase_count,total_amount
3929,3c8d541112a74af99e88abbd0692f00e,8,1608.69
15693,f1d65ae63f174b8f80fa063adcaa63b7,13,1365.66
11422,ae6f43089b674728a50b8727252d3305,16,1327.74
6366,626df8678e2a4953b9098246418c9cfa,13,1321.42
7492,73afdeca19e349b98f09e928644610f8,10,1319.97
5358,52959f19113e4241a8cb3bef486c6412,12,1292.86
11334,ad1f0a409ae642bc9a43f31f56c130fc,5,1258.19
13672,d240308de0ee4cf8bb6072816268582b,14,1251.99
9673,946fc0d3ecc4492aa4cc06cf6b1492c3,17,1232.40
6484,6406abad8e2c4b8584e4f68003de148d,12,1211.76


5번 갔는데 금액이 높은사람도 보임 그래서 금액상위랑 회수를 같이 보는게 맞는거 같음

In [24]:
customer.describe()

,purchase_count,total_amount
count,16578.000000,16578.000000
mean,8.381771,107.096874
std,5.009822,126.393939
min,1.000000,0.050000
25%,5.000000,23.682500
50%,7.000000,72.410000
75%,11.000000,150.937500
max,36.000000,1608.690000


그래서 vip고객은 상위20% and 구매횟수 20%로 잡고 신규고객은 하위 30%로 잡은뒤 나머지남은 50%는 기존고객으로 세그먼트 나눌 생각

In [25]:
# 분위수(quantile)를 활용 처음봄.... 백분위로 상위20이랑 하위30 나눠봤음...
amount_top20 = customer['total_amount'].quantile(0.8)
count_top20 = customer['purchase_count'].quantile(0.8)
count_bottom30 = customer['purchase_count'].quantile(0.3)

In [26]:
def segment_customer(row):
    # VIP 조건
    if (row['total_amount'] >= amount_top20) and (row['purchase_count'] >= count_top20):
        return 'VIP'
    # 신규 조건
    elif row['purchase_count'] <= count_bottom30:
        return '신규'
    # 나머지
    else:
        return '기존'

In [27]:
customer['segment'] = customer.apply(segment_customer, axis=1)

In [28]:
customer['segment'].value_counts()

segment
기존     9380
신규     5623
VIP    1575
Name: count, dtype: int64

In [29]:
customer.groupby('segment')[['purchase_count', 'total_amount']].mean()

,purchase_count,total_amount
segment,,
VIP,15.234921,297.475860
기존,10.130384,109.218045
신규,3.545261,50.233368


어떤 기간내에 이만큼 삿다?

# 시간 기준으로 각고객들이 마지막으로 구매한 시점

In [30]:
transaction.head()

,Unnamed: 0,person,event,value,time,amount
12654,12654,02c083884c7d45b39cc68e1314fec56c,transaction,{'amount': 0.8300000000000001},0,0.83
12657,12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,transaction,{'amount': 34.56},0,34.56
12659,12659,54890f68699049c2a04d415abc25e717,transaction,{'amount': 13.23},0,13.23
12670,12670,b2f1cd155b864803ad8334cdf13c4bd2,transaction,{'amount': 19.51},0,19.51
12671,12671,fe97aa22dd3e48c8b143116a8403dd52,transaction,{'amount': 18.97},0,18.97


In [31]:
customer['last_purchase'] = transaction.groupby('person')['time'].max()

In [32]:
customer.head()

,person,purchase_count,total_amount,segment,last_purchase
0,0009655768c64bdeb2e877511632db8f,8,127.60,기존,NaN
1,00116118485d4dfda04fdbaba9a87b5c,3,4.09,신규,NaN
2,0011e0d4e6b944f998e987f904e8c1e5,5,79.46,신규,NaN
3,0020c2b971eb4e9188eac86d93036a77,8,196.86,기존,NaN
4,0020ccbbb6d84e358d3414a3ff76cffd,12,154.05,기존,NaN


In [33]:
customer['last_purchase'].isna().sum()

np.int64(16578)

?????????? 

In [34]:
last_purchase = transaction.groupby('person')['time'].max()
customer['last_purchase'] = customer['person'].map(last_purchase)

In [35]:
print(customer[['person', 'last_purchase']].head())
print(customer['last_purchase'].isna().sum())

                             person  last_purchase
0  0009655768c64bdeb2e877511632db8f            696
1  00116118485d4dfda04fdbaba9a87b5c            474
2  0011e0d4e6b944f998e987f904e8c1e5            654
3  0020c2b971eb4e9188eac86d93036a77            708
4  0020ccbbb6d84e358d3414a3ff76cffd            672
0


고객별 최근 구매 시점 추가

그래서 시간을 백분위로 나눠 상위30퍼는 최근활동 하위 30퍼는 오래동안 안찾아온 고객 나머지는 보통으로 나눌 생각

In [36]:
recency_top30 = customer['last_purchase'].quantile(0.7)
recency_bottom30 = customer['last_purchase'].quantile(0.3)

def recency_segment(row):
    if row['last_purchase'] >= recency_top30:
        return '최근활동'
    elif row['last_purchase'] <= recency_bottom30:
        return '오래쉼'
    else:
        return '보통'

customer['recency_segment'] = customer.apply(recency_segment, axis=1)

In [37]:
customer.head()

,person,purchase_count,total_amount,segment,last_purchase,recency_segment
0,0009655768c64bdeb2e877511632db8f,8,127.60,기존,696,최근활동
1,00116118485d4dfda04fdbaba9a87b5c,3,4.09,신규,474,오래쉼
2,0011e0d4e6b944f998e987f904e8c1e5,5,79.46,신규,654,보통
3,0020c2b971eb4e9188eac86d93036a77,8,196.86,기존,708,최근활동
4,0020ccbbb6d84e358d3414a3ff76cffd,12,154.05,기존,672,보통


그래서 이걸 vip/기존/신규로 나눠서 계산을 해봐야 되는데.....

그러니깐 지금 time컬럼 값이 시간단위로 표시되서 24시간이 1일 총 694면?? 28일 22시간은 무슨시점으로 나타난걸까?? ..... 지금 당장 이컬럼으로 할수가 없다...

# 튜터님 피드백
고객 세그먼트 설정 방향은 전체적으로 나쁘지 않습니다. 다만 이런 지표를 만들 때 가장 먼저 고려해야 할 것은 “왜 이 기준으로 나누었는가”에 대한 근거입니다. 세그먼트 기준을 그렇게 결정한 이유가 얼마나 객관적이고 데이터 기반인지에 따라 이후 분석 결과의 타당성이 크게 달라질 수 있습니다.

특히 마케팅 데이터에서 고객 충성도를 비교적 간단하고 널리 쓰이는 방식으로 분석할 때는 RFM 기준을 많이 활용합니다.

Frequency: 구매 횟수
Monetary: 구매 금액
Recency: 최근 구매 시점

현재처럼 구매 관련 지표를 활용하려는 방향은 좋지만, 가능하다면 이러한 RFM 기반 지표로 고객 충성도를 정의하고 세그먼트를 구성하는 방식이 더 일반적이고 설득력 있습니다. 이후 고객 반응, 전환, 프로모션 효과 분석까지 연결하기에도 훨씬 유리합니다.

다시 vip/기존/신규 다시 데이터기반으로 재정비

In [38]:
customer.describe()

,purchase_count,total_amount,last_purchase
count,16578.000000,16578.000000,16578.000000
mean,8.381771,107.096874,628.574738
std,5.009822,126.393939,82.766010
min,1.000000,0.050000,12.000000
25%,5.000000,23.682500,594.000000
50%,7.000000,72.410000,654.000000
75%,11.000000,150.937500,690.000000
max,36.000000,1608.690000,714.000000


기술통계를 찍어본결과 25%값 5랑 75%값으로 기준을나눠 저활동 일반 vip로 나눠도 될거 같음.

In [ ]:
# 기존꺼 값 포함 다시 만든 vip 기준
def freq_segment(x):
    if x <= 5:
        return '저활동'
    elif x <= 11:
        return '중간'
    else:
        return '고활동'

def money_segment(x):
    if x <= 23:
        return '저금액'
    elif x <= 150:
        return '중간금액'
    else:
        return '고금액'

customer['freq_segment_new'] = customer['purchase_count'].apply(freq_segment)
customer['money_segment_new'] = customer['total_amount'].apply(money_segment)

customer['segment_new'] = customer['freq_segment_new'] + '_' + customer['money_segment_new']

customer['final_segment_new'] = '일반'
customer.loc[
    (customer['freq_segment_new'] == '고활동') &
    (customer['money_segment_new'] == '고금액'),
    'final_segment_new'
] = 'VIP'

In [40]:
customer.head()

,person,purchase_count,total_amount,segment,last_purchase,recency_segment,freq_segment_new,money_segment_new,segment_new,final_segment_new
0,0009655768c64bdeb2e877511632db8f,8,127.60,기존,696,최근활동,중간,중간금액,중간_중간금액,일반
1,00116118485d4dfda04fdbaba9a87b5c,3,4.09,신규,474,오래쉼,저활동,저금액,저활동_저금액,일반
2,0011e0d4e6b944f998e987f904e8c1e5,5,79.46,신규,654,보통,저활동,중간금액,저활동_중간금액,일반
3,0020c2b971eb4e9188eac86d93036a77,8,196.86,기존,708,최근활동,중간,고금액,중간_고금액,일반
4,0020ccbbb6d84e358d3414a3ff76cffd,12,154.05,기존,672,보통,고활동,고금액,고활동_고금액,VIP
